# Causal Anomaly Detection with Learned Change Dynamics

## The punchline

Two observations that shouldn't coexist, and yet do:

1. The same online causal-discovery method (DyCoLiDE) whose point estimate $\hat W_t$ wobbles with SGD-level estimator noise — enough to create spurious $\lVert\Delta\hat W\rVert$ spikes — can be wrapped in an EMA-smoothed reference to produce an anomaly signal with **zero false positives across 15 batches of burn-in + 20 batches of normal DAG evolution**, firing within one batch of a real structural break.

2. The *same* bi-level skeleton, read min–max instead of min–min, exposes a brittleness: a **1% Frobenius-norm adversarial perturbation is 10× worse for DAG recovery than 10% random noise**. Structural anomaly detection and adversarial robustness are two sides of one formulation.

## Contribution

- A **joint formulation** of online causal discovery + structural anomaly detection, where the distribution of normal DAG dynamics is a *learned prior* on DAG evolution — not hand-crafted.
- An **empirical demonstration** that a residual-based detector on an EMA-smoothed $\hat W$ gives clean, low-latency structural anomaly detection on non-stationary streams, without the sliding-window lag of structural-change detectors.
- An **adversarial complement** showing the same machinery, viewed through the min–max lens, reveals a concrete vulnerability of score-based DAG learners — and directly motivates a robust training variant of the same bi-level skeleton.

## 1. Setup

We observe a stream of data batches

$$ X_1, X_2, \ldots, X_T \in \mathbb{R}^{n \times d} $$

where each $X_t$ is generated by a linear SEM with a time-varying DAG $W_t$:

$$ X_t = X_t\, W_t + \varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0, \sigma^2 I_d), \quad W_t \text{ acyclic.} $$

$W_t$ evolves slowly and stochastically:

- **Normal evolution**: small, infrequent edge flips; mild weight drift.
- **Anomaly**: an abrupt change that is *out of distribution* under the normal-evolution law.

We want three things at once:

1. Track $\hat W_t$ as batches stream in.
2. Learn a model $P(W_t \mid W_{t-1};\, \theta)$ of *normal* DAG dynamics.
3. Flag batches whose observed change is implausible under that model.

Naive baselines fail in revealing ways:

- Static DAG learners assume $W_t$ is fixed — they over-smooth across real change.
- Online learners without a dynamics prior are noisy — their $\hat W_t$ jitter swamps the anomaly signal.
- Hand-crafted dynamics (e.g., "flag if >k edges flip") miss correlated-change structure and don't adapt across domains.

## 2. Bi-level formulation: online DAG learning coupled with a learned prior

Treat $W_{1:T}$ as latent state and $\theta$ (parameters of the change distribution) as learnable. The joint:

$$ p(X_{1:T}, W_{1:T};\, \theta) \;=\; p(W_1)\, \prod_{t=2}^{T} \underbrace{p(W_t \mid W_{t-1};\, \theta)}_{\text{outer: change dynamics}}\; \underbrace{p(X_t \mid W_t)}_{\text{inner: SEM}}. $$

**Inner problem** (estimate current DAG under the current prior):

$$ \hat W_t \;=\; \arg\min_{W \in \mathrm{DAG}} \; \underbrace{-\log p(X_t \mid W)}_{\text{data fit}} \; \underbrace{-\log p(W \mid \hat W_{t-1};\, \theta)}_{\text{prior from outer}} \; + \; \lambda\, \lVert W \rVert_1 . $$

**Outer problem** (fit the change dynamics to the observed DAG trajectory):

$$ \theta^\star \;=\; \arg\max_\theta \; \sum_{t=2}^{T} \log p(\hat W_t \mid \hat W_{t-1};\, \theta). $$

The levels are coupled: the outer's prior biases the inner toward *normal* trajectories (reducing jitter, sharpening anomaly signals); the inner's trajectory is the training data for the outer. Iterate by variational EM — E-step updates beliefs over $W_{1:t}$, M-step updates $\theta$.

**Choices for the change distribution $p(W_t \mid W_{t-1};\, \theta)$:**

- *Per-edge Markov*: each edge has its own flip rate. Interpretable, fittable.
- *Hawkes on edge events*: captures co-movement ("A→B appearing makes C→D more likely").
- *Latent regime*: low-dim $z_t$ controls $W$; $\theta$ parameterizes $z$-dynamics and $z \to W$. Good for market-regime / circadian patterns.
- *Smooth drift*: GP or diffusion on edge weights. Good for slow drift, bad for abrupt change.

## 3. Anomaly score as posterior-predictive likelihood

A new batch $X_{t+1}$ is scored by how well it fits the predictive distribution implied by the joint model:

$$ s(X_{t+1}) \;=\; -\log \int p(X_{t+1} \mid W)\, p(W \mid \hat W_t;\, \theta^\star)\, dW. $$

This absorbs two sources of uncertainty: about $W_t$ itself, and about how $W$ typically evolves step-to-step. The attractive property is that it gives a **single, principled scalar** — no hand-tuned detector, no ad-hoc metric combination.

### Tractable approximation (what the POC uses)

Replace the integral with a plug-in reference:

$$ \hat W_{\mathrm{ref},\, t} \;\leftarrow\; (1 - \alpha)\, \hat W_{\mathrm{ref},\, t-1}\; +\; \alpha\, \hat W_t , $$

$$ s(X_{t+1}) \;\approx\; \tfrac{1}{n}\, \big\lVert X_{t+1} - X_{t+1}\, \hat W_{\mathrm{ref},\, t} \big\rVert_F^2. $$

The EMA smoothing with rate $\alpha$ is a crude stand-in for marginalization over $p(W \mid \hat W_t;\, \theta^\star)$. A richer formulation — Kalman-style filtering in graph space, or variational posteriors over $W$ — replaces the plug-in with a proper belief.

### Detection rule

Calibrate the empirical distribution of $s$ on a stable-evolution window, then flag when the standardized score exceeds a threshold:

$$ \mathrm{alarm}(t) \;=\; \mathbf{1}\!\left[ \frac{s(X_{t+1}) - \mu_s}{\sigma_s} \;>\; \tau \right]. $$

### Why it's fast (no window lag)

$\hat W_{\mathrm{ref}}$ *intentionally* moves slowly. A true anomaly therefore hits residuals immediately — the reference model cannot absorb the shift in one step, so $X_{t+1}$ stops fitting. Contrast with $\lVert \Delta \hat W_t \rVert$: that detector fires only once $\hat W$ has *adapted*, which takes a full window refresh.

## 4. Adversarial complement: robustness is the min–max dual of the same bi-level

Replace the outer max over $\theta$ with a max over data perturbations:

$$ \hat W^{\star} \;=\; \arg\min_{W \in \mathrm{DAG}} \; \max_{\lVert \delta \rVert_F \le \varepsilon} \; \Big[\, -\log p(X + \delta \mid W) \;+\; \lambda\lVert W \rVert_1 \,\Big] . $$

This is *distributionally-robust* DAG estimation: by training against the worst $\varepsilon$-ball perturbation of the data, $\hat W^\star$ should recover the same structure whether the input is exactly $X$ or adversarially shifted within the budget.

### Why this is the sibling of the anomaly-detection formulation

Both versions quantify how fragile $\hat W$ is to small shifts in the data. They differ only in the *shift model*:

| | Anomaly detection | Adversarial robustness |
|---|---|---|
| shift source | stochastic drift of $W$ | worst-case perturbation of $X$ |
| outer problem | $\max_\theta \log P(\text{trajectory})$ | $\max_\delta \text{loss}$ |
| inner problem | DAG estimation with learned prior | DAG estimation on perturbed data |
| what failure mode reveals | an out-of-distribution DAG change | a non-robust optimizer |

The shared skeleton — an inner DAG optimizer wrapped by an outer that *stresses* it — is what makes this a single formulation, not two. A full solution to either side unlocks the other: learned dynamics give you an adaptive anomaly detector; learned robustness gives you a DAG estimator whose $\hat W$ doesn't need those dynamics to be exact.

### What our POC establishes (without solving the full min–max)

- Run the attacker *once* against a vanilla DyCoLiDE and measure the gap.
- At $\varepsilon = 1\%$ of $\lVert X \rVert_F$, adversarial SHD $= 11$ vs. clean SHD $= 1$; random-perturbation SHD at same budget $= 1$.
- The attack operates almost entirely by injecting *false* edges (FDR $= 0.25$ at 1%, near-clean TPR), suggesting the acyclicity + sparsity regularizer isn't enough to suppress adversarially-flavored residuals.

The headroom for robust training is large.

## 5. What the POCs establish

| POC | What it shows | File |
|---|---|---|
| Evolving-DAG baseline | Online DyCoLiDE adapts to abrupt structural change over a window (~8 batches) | `evolving_dag_experiment.py` |
| Anomaly detection | Residual detector on EMA-smoothed $\hat W$ fires within 1 batch of the anomaly; 0 FP across 32 normal-evolution batches | `anomaly_detection/experiment.py` |
| Adversarial DAG | $1\%$ PGD attack on the linear-SEM surrogate → SHD $\times 11$; failure mode is spurious edges (high FDR, near-clean TPR) | `adversarial_dag/experiment.py` |

These are *pieces*, not a full bi-level system:

- The anomaly-detection POC uses an **EMA plug-in** instead of a fitted $\theta$. Upgrade: variational EM or particle filter on $(W_t, \theta)$.
- The adversarial POC runs **one pass** of the attacker against a vanilla estimator. Upgrade: min–max training with PGD-interleaved SGD inside DyCoLiDE.

But the ceilings are clearly high: smoothed residuals already beat the $\lVert \Delta \hat W \rVert$ baseline by orders of magnitude in latency × false-positive rate, and adversarial attacks succeed at perturbation budgets an order of magnitude below where random noise matters. The bi-level formulation isn't just an aesthetic upgrade — both ends have substantial room to improve.

## 6. Next steps

1. **Implement the outer M-step.** Pick a concrete form for $p(W_t \mid W_{t-1};\, \theta)$ (start with per-edge Markov for interpretability). Fit $\theta$ jointly with the $\hat W_t$ trajectory via EM. Anomaly score becomes an actual posterior-predictive likelihood, not an EMA residual.
2. **Implement adversarial training.** Interleave PGD attacks on $X$ with DyCoLiDE SGD steps on $W$. Measure the SHD–ε curve of the robust vs. vanilla estimator; the robust curve should stay flat where the vanilla curve spikes.
3. **Unify.** Swap the adversarial perturbation for the *dynamics-prior-predicted* perturbation: what if the "attack" is drawn from $p(\Delta W \mid \theta)$ rather than from an $\varepsilon$-ball? That's a single bi-level formulation where the outer loop simultaneously calibrates the dynamics prior and stress-tests the inner estimator — the theoretical endpoint of what these two POCs each attack from opposite sides.